In [264]:
from pathlib import Path
import json
import re
from datetime import datetime

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [265]:
SILVER_FOLDER = Path("../../Data/silver")
SILVER_NLP_FOLDER = Path("../../Data/silver_nlp")
GOLD_FOLDER = Path("../../Data/gold")

GOLD_FOLDER.mkdir(parents=True, exist_ok=True)

In [266]:
# pip install transformers torch sentencepiece

In [267]:
SUMMARY_MODELS = {
    "nl": "yhavinga/t5-v1.1-base-dutch-cnn-test",
    "en": "facebook/bart-large-cnn",
    "default": "google/flan-t5-base"
}

PROMPTS = {
    "nl": (
        "Vat de onderstaande tekst feitelijk samen in het Nederlands.\n"
        "Gebruik alleen informatie die letterlijk in de tekst staat.\n"
        "Verzin geen bronnen, websites, organisaties, personen, jaartallen of cijfers.\n"
        "Schrijf zakelijk en neutraal.\n\n"
        "TEKST:\n"
    ),
    "en": (
        "Summarize the following text factually in English.\n"
        "Use only information explicitly present in the text.\n"
        "Do not invent sources, websites, organizations, people, dates, or numbers.\n"
        "Write in a neutral report style.\n\n"
        "TEXT:\n"
    ),
    "default": (
        "Summarize the following text factually.\n"
        "Use only information explicitly present in the text.\n\n"
        "TEXT:\n"
    )
}

BAD_SUMMARY_PATTERNS = [
    "summarize the text",
    "summarize the following text",
    "use only information",
    "do not invent",
    "write in a neutral",
    "text:",
    "tekst:",
    "newsquiz",
    "http://",
    "https://",
    "www.",
    "cnn",
    "nu.nl zet op een rij",
    "further information",
    "methodology can be found",
    "nu.nl",
    "facebook",
    "bewering:",
    "wekelijkse rubriek",
    "belangrijkste nieuws",
    "nieuws van de dag",
    "vat de onderstaande",
    "gebruik alleen informatie",
    "verzin geen",
    "schrijf zakelijk",
    "tekst:",
    "text:",
    "feitel",
    "gebronnen"
]

In [268]:
def looks_like_contact_or_address_line(line):
    lower = line.lower()

    if re.search(r"\b\d{4}\s?[a-z]{2}\b", lower):
        return True

    if re.search(r"\b(postbus|postcode|telefoon|tel\.?|t\s+\d|email|e-mail|info@|www\.)\b", lower):
        return True

    if re.search(r"@[a-z0-9.-]+\.[a-z]{2,}", lower):
        return True

    if re.search(r"\b\d{2,}\b", lower) and len(line.split()) <= 6:
        return True

    return False

In [269]:
def choose_summary_model(language):
    return SUMMARY_MODELS.get(language, SUMMARY_MODELS["default"])

In [270]:
def load_model(language):
    model_name = choose_summary_model(language)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    return model_name, tokenizer, model

In [271]:
def clean_input_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    text = re.sub(
        r"Summarize the.*?(TEXT:|TEKST:)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(
        r"Vat de onderstaande.*?(TEXT:|TEKST:)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return text.strip()

In [272]:
def clean_summary_text(summary):
    summary = summary.strip()
    summary = re.sub(r"\s+", " ", summary)

    summary = re.sub(
        r"Summarize the.*?(TEXT:|TEKST:)",
        "",
        summary,
        flags=re.IGNORECASE | re.DOTALL
    )

    summary = re.sub(
        r"Vat de onderstaande.*?(TEXT:|TEKST:)",
        "",
        summary,
        flags=re.IGNORECASE | re.DOTALL
    )

    summary = re.sub(r"http\S+|www\.\S+", "", summary)

    return summary.strip()

In [273]:
def is_bad_summary(summary):
    if not summary:
        return True

    if len(summary.split()) < 20:
        return True

    lower = summary.lower()

    if any(pattern in lower for pattern in BAD_SUMMARY_PATTERNS):
        return True

    return False

In [274]:
def is_title_or_metadata_chunk(text):
    lower = text.lower()
    words = len(text.split())

    metadata_terms = [
        "sector summary",
        "part 1",
        "part 2",
        "table of contents",
        "inhoudsopgave",
        "copyright",
        "all rights reserved",
        "november",
        "january",
        "february",
        "march",
        "april",
        "may",
        "june",
        "july",
        "august",
        "september",
        "october",
        "december"
    ]

    return words < 80 and any(term in lower for term in metadata_terms)

In [275]:
def get_summary_length(text, ratio=0.30, min_tokens=60, max_tokens=180):
    word_count = len(text.split())
    target = int(word_count * ratio)

    return {
        "min": max(30, min(min_tokens, int(target * 0.6))),
        "max": min(max_tokens, max(min_tokens, target))
    }

In [276]:
def summarize_text(
    text,
    language,
    tokenizer,
    model,
    ratio=0.30,
    max_tokens=180
):
    text = clean_input_text(text)

    if not text or len(text.split()) < 40:
        return ""

    length = get_summary_length(
        text=text,
        ratio=ratio,
        min_tokens=60,
        max_tokens=max_tokens
    )

    prompt = PROMPTS.get(language, PROMPTS["default"]) + text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    output_ids = model.generate(
        **inputs,
        max_new_tokens=length["max"],
        min_new_tokens=length["min"],
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.4,
        do_sample=False,
        early_stopping=True
    )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return clean_summary_text(summary)

In [277]:
def summarize_chunks(
    chunks,
    language,
    tokenizer,
    model,
    max_chunks=30
):
    chunk_summaries = []

    for chunk in chunks[:max_chunks]:
        chunk_id = chunk["chunk_id"]
        text = chunk["text"]

        if is_title_or_metadata_chunk(text):
            print(f"Skipped title/metadata chunk {chunk_id}")
            continue

        if len(text.split()) < 40:
            continue

        summary = summarize_text(
            text=text,
            language=language,
            tokenizer=tokenizer,
            model=model,
            ratio=0.25,
            max_tokens=160
        )

        summary = clean_summary_text(summary)

        if is_bad_summary(summary):
            print(f"Skipped bad summary for {chunk_id}")
            continue

        chunk_summaries.append({
            "chunk_id": chunk_id,
            "summary": summary
        })

    return chunk_summaries

In [278]:
def classify_summary_role(summary):
    lower = summary.lower()

    context_terms = [
        "context", "background", "population", "region", "sector",
        "project", "programme", "program", "assessment", "industry",
        "education", "research", "market", "organisation", "organization",
        "achtergrond", "context", "bevolking", "regio", "sector",
        "project", "programma", "verkenning", "onderzoek", "onderwijs",
        "markt", "organisatie"
    ]

    need_terms = [
        "risk", "challenge", "problem", "need", "shortage", "pressure",
        "vulnerable", "impact", "affected", "requires", "demand",
        "transition", "change", "threat", "gap",
        "risico", "uitdaging", "probleem", "behoefte", "tekort",
        "druk", "kwetsbaar", "impact", "gevolg", "vraagt",
        "transitie", "verandering", "dreiging", "kloof"
    ]

    approach_terms = [
        "approach", "monitor", "plan", "strategy", "partnership",
        "programme", "program", "support", "develop", "provide",
        "measures", "management", "implementation", "training",
        "collaboration",
        "aanpak", "monitoren", "plan", "strategie", "samenwerking",
        "programma", "ondersteunen", "ontwikkelen", "aanbieden",
        "maatregelen", "beheer", "uitvoering", "training"
    ]

    example_terms = [
        "for example", "including", "such as", "examples", "among others",
        "bijvoorbeeld", "zoals", "waaronder", "onder meer"
    ]

    outcome_terms = [
        "goal", "aim", "contribution", "result", "expected", "prioritise",
        "prioritize", "improve", "reduce", "strengthen", "enable",
        "support", "prepare",
        "doel", "bijdrage", "resultaat", "verwacht", "prioriteren",
        "verbeteren", "verminderen", "versterken", "mogelijk maken",
        "ondersteunen", "voorbereiden"
    ]

    scores = {
        "context": sum(term in lower for term in context_terms),
        "need": sum(term in lower for term in need_terms),
        "approach": sum(term in lower for term in approach_terms),
        "examples": sum(term in lower for term in example_terms),
        "outcome": sum(term in lower for term in outcome_terms)
    }

    return max(scores, key=scores.get)

In [279]:
def group_summaries_by_role(chunk_summaries):
    grouped = {
        "context": [],
        "need": [],
        "approach": [],
        "examples": [],
        "outcome": [],
        "other": []
    }

    for item in chunk_summaries:
        summary = item["summary"]

        if is_bad_summary(summary):
            continue

        role = classify_summary_role(summary)

        if role in grouped:
            grouped[role].append(summary)
        else:
            grouped["other"].append(summary)

    return grouped

In [280]:
def take_first_sentences(text, max_sentences=2):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    return " ".join(sentences[:max_sentences])

In [281]:
def build_paragraph_from_group(group, max_items=2, max_sentences_each=2):
    selected = group[:max_items]

    parts = [
        take_first_sentences(item, max_sentences=max_sentences_each)
        for item in selected
    ]

    paragraph = " ".join(part for part in parts if part)
    paragraph = re.sub(r"\s+", " ", paragraph)

    return paragraph.strip()

In [282]:
def postprocess_final_summary(summary):
    summary = summary.replace("•", "")

    # Remove citation leftovers like "1970 12"
    summary = re.sub(r"\b(\d{4})\s+\d{1,2}\b", r"\1", summary)

    # Normalize whitespace
    summary = re.sub(r"[ \t]+", " ", summary)
    summary = re.sub(r"\n{3,}", "\n\n", summary)

    return summary.strip()

In [283]:
def create_english_structured_summary(chunk_summaries):
    grouped = group_summaries_by_role(chunk_summaries)

    paragraphs = []

    context = build_paragraph_from_group(
        grouped["context"] or grouped["other"],
        max_items=2
    )

    need = build_paragraph_from_group(
        grouped["need"],
        max_items=2
    )

    approach = build_paragraph_from_group(
        grouped["approach"],
        max_items=2
    )

    examples = build_paragraph_from_group(
        grouped["examples"],
        max_items=2
    )

    outcome = build_paragraph_from_group(
        grouped["outcome"],
        max_items=2
    )

    if context:
        paragraphs.append(context)

    if need:
        paragraphs.append(need)

    if approach:
        paragraphs.append(approach)

    if examples:
        paragraphs.append(examples)

    if outcome:
        paragraphs.append(outcome)

    if len(paragraphs) < 3:
        fallback = [
            item["summary"]
            for item in chunk_summaries
            if not is_bad_summary(item["summary"])
        ]
        paragraphs = [
            build_paragraph_from_group(fallback[:2]),
            build_paragraph_from_group(fallback[2:4]),
            build_paragraph_from_group(fallback[4:6])
        ]

    final_summary = "\n\n".join(p for p in paragraphs if p)
    return postprocess_final_summary(final_summary)

In [284]:
def create_dutch_structured_summary(chunk_summaries):
    grouped = group_summaries_by_role(chunk_summaries)

    paragraphs = []

    context = build_paragraph_from_group(
        grouped["context"] or grouped["other"],
        max_items=2
    )

    need = build_paragraph_from_group(
        grouped["need"],
        max_items=2
    )

    approach = build_paragraph_from_group(
        grouped["approach"],
        max_items=2
    )

    examples = build_paragraph_from_group(
        grouped["examples"],
        max_items=2
    )

    outcome = build_paragraph_from_group(
        grouped["outcome"],
        max_items=2
    )

    if context:
        paragraphs.append(context)

    if need:
        paragraphs.append(need)

    if approach:
        paragraphs.append(approach)

    if examples:
        paragraphs.append(examples)

    if outcome:
        paragraphs.append(outcome)

    if len(paragraphs) < 3:
        fallback = [
            item["summary"]
            for item in chunk_summaries
            if not is_bad_summary(item["summary"])
        ]
        paragraphs = [
            build_paragraph_from_group(fallback[:2]),
            build_paragraph_from_group(fallback[2:4]),
            build_paragraph_from_group(fallback[4:6])
        ]

    final_summary = "\n\n".join(p for p in paragraphs if p)
    return postprocess_final_summary(final_summary)

In [285]:
def create_final_summary(
    chunk_summaries,
    language,
    tokenizer,
    model
):
    valid_summaries = [
        item
        for item in chunk_summaries
        if not is_bad_summary(item["summary"])
    ]

    if not valid_summaries:
        return ""

    if language == "nl":
        return create_dutch_structured_summary(valid_summaries)

    if language == "en":
        return create_english_structured_summary(valid_summaries)

    return create_english_structured_summary(valid_summaries)

In [286]:
def get_modeling_hints(nlp_data):
    return {
        "main_topics": nlp_data.get("modeling_hints", {}).get("main_topics", []),
        "important_entities": nlp_data.get("modeling_hints", {}).get("important_entities", [])
    }

In [287]:
def load_modeling_hints(document_id):
    nlp_file = SILVER_NLP_FOLDER / f"{document_id}_nlp.json"

    if not nlp_file.exists():
        return {
            "main_topics": [],
            "important_entities": []
        }

    nlp_data = json.loads(nlp_file.read_text(encoding="utf-8"))

    return get_modeling_hints(nlp_data)

In [288]:
def save_gold_output(
    document_id,
    language,
    final_summary,
    chunk_summaries,
    modeling_hints,
    model_name
):
    output_path = GOLD_FOLDER / f"{document_id}_gold.json"

    gold_data = {
        "document_id": document_id,
        "language": language,
        "summary": final_summary,
        "chunk_summaries": chunk_summaries,
        "modeling_hints_used": modeling_hints,
        "model": {
            "task": "local_summarization",
            "name": model_name,
            "type": "local_open_source"
        },
        "processed_at": datetime.now().isoformat()
    }

    output_path.write_text(
        json.dumps(gold_data, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"Saved Gold output: {output_path}")

In [289]:
def process_single_document(silver_file):
    document_id = silver_file.stem.replace("_silver", "")

    print(f"Generating summary for {document_id}...")

    silver_data = json.loads(silver_file.read_text(encoding="utf-8"))

    language = silver_data.get("language", "default")
    chunks = silver_data.get("chunks", [])

    if not chunks:
        print(f"No chunks found for {document_id}. Skipping.")
        return

    model_name, tokenizer, model = load_model(language)

    modeling_hints = load_modeling_hints(document_id)

    chunk_summaries = summarize_chunks(
        chunks=chunks,
        language=language,
        tokenizer=tokenizer,
        model=model
    )

    final_summary = create_final_summary(
        chunk_summaries=chunk_summaries,
        language=language,
        tokenizer=tokenizer,
        model=model
    )

    save_gold_output(
        document_id=document_id,
        language=language,
        final_summary=final_summary,
        chunk_summaries=chunk_summaries,
        modeling_hints=modeling_hints,
        model_name=model_name
    )

In [290]:
def run_gold_layer():
    silver_files = sorted(SILVER_FOLDER.glob("doc_*_silver.json"))

    if not silver_files:
        print("No silver files found.")
        return

    for silver_file in silver_files:
        process_single_document(silver_file)

    print("Gold layer completed.")

In [291]:
run_gold_layer()

Generating summary for doc_01...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6631.50it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=160) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=60) and `min_length`(=75) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://h

Skipped bad summary for chunk_003


Both `max_new_tokens` (=97) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=58) and `min_length`(=75) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved Gold output: ..\..\Data\gold\doc_01_gold.json
Gold layer completed.
